# Econometric Analysis: EDA-Style Pooled Model (No Fixed Effects)

This notebook estimates the same AI-exposure-by-year relationship used for EDA-style interpretation, in a standalone output file.

Model:
$$
\log(wage_{it}) = \alpha + \sum_{t=2021}^{2024} \beta_t (AI_i \times 1\{year=t\}) + \delta_t + u_{it}
$$

- No individual fixed effects
- Clustered standard errors at individual level
- Intended as descriptive/pooled benchmark against TWFE

In [5]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm

pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('ggplot')

In [6]:
project_root = Path.cwd()
if project_root.name == 'output':
    project_root = project_root.parent

data_path = project_root / 'data' / 'clean' / '08_wages_ai_analysis_panel.csv'
eda_coef_out = project_root / 'output' / 'panel_eda_style_coefficients.csv'
eda_summary_out = project_root / 'output' / 'panel_eda_style_summary.txt'

raw = pd.read_csv(data_path)
print('Rows:', len(raw))
print('People:', raw['person_id'].nunique())
print('Years:', sorted(raw['year'].dropna().unique().tolist()))

Rows: 45147
People: 14399
Years: [2020, 2021, 2022, 2023, 2024]


## EDA-Style Model in Output (No Fixed Effects)

To mirror the EDA relationship check, estimate a pooled model without individual fixed effects:

$$
\log(wage_{it}) = \alpha + \sum_{t=2021}^{2024} \beta_t (AI_i \times 1\{year=t\}) + \delta_t + u_{it}
$$

This keeps the same interaction structure but does not absorb person-level time-invariant heterogeneity.

In [7]:
import statsmodels.formula.api as smf

eda_df = raw.copy()
eda_df['year'] = pd.to_numeric(eda_df['year'], errors='coerce')
eda_df['log_wage'] = pd.to_numeric(eda_df['log_wage'], errors='coerce')
eda_df['ai_exposure'] = pd.to_numeric(eda_df['ai_exposure'], errors='coerce')
eda_df = eda_df.dropna(subset=['person_id', 'year', 'log_wage', 'ai_exposure']).copy()
eda_df = eda_df[eda_df['year'].between(2020, 2024)].copy()
eda_df['year'] = eda_df['year'].astype(int)

# EDA-style pooled interaction model (no individual FE)
eda_model = smf.ols(
    'log_wage ~ C(year) + ai_exposure:C(year)',
    data=eda_df
).fit(cov_type='cluster', cov_kwds={'groups': eda_df['person_id']})

print('EDA-style pooled model estimated (no FE).')
print(f'Observations: {len(eda_df):,}')
print(f'Individuals: {eda_df["person_id"].nunique():,}')

terms = [
    'ai_exposure:C(year)[2020]',
    'ai_exposure:C(year)[2021]',
    'ai_exposure:C(year)[2022]',
    'ai_exposure:C(year)[2023]',
    'ai_exposure:C(year)[2024]',
]

eda_coef_table = pd.DataFrame({
    'coef': eda_model.params.reindex(terms),
    'std_err': eda_model.bse.reindex(terms),
    'p_value': eda_model.pvalues.reindex(terms),
    'ci_low': eda_model.conf_int().reindex(terms)[0],
    'ci_high': eda_model.conf_int().reindex(terms)[1],
}).round(6)

eda_coef_table.index = ['slope_2020', 'slope_2021', 'slope_2022', 'slope_2023', 'slope_2024']
eda_coef_table

EDA-style pooled model estimated (no FE).
Observations: 44,577
Individuals: 14,296


,coef,std_err,p_value,ci_low,ci_high
slope_2020,0.5850,0.0354,0.0000,0.5156,0.6545
slope_2021,0.6070,0.0354,0.0000,0.5377,0.6763
slope_2022,0.6292,0.0360,0.0000,0.5587,0.6997
slope_2023,0.7171,0.0376,0.0000,0.6433,0.7908
slope_2024,0.7720,0.0361,0.0000,0.7012,0.8429


In [8]:
eda_coef_table.to_csv(eda_coef_out)

with eda_summary_out.open('w', encoding='utf-8') as f:
    f.write('EDA-style pooled model (no individual fixed effects)\n')
    f.write(f'Observations: {len(eda_df):,}\n')
    f.write(f'Individuals: {eda_df["person_id"].nunique():,}\n\n')
    f.write('Year-specific AI-exposure slopes:\n')
    f.write(eda_coef_table.to_string())
    f.write('\n\nFull model summary:\n')
    f.write(str(eda_model.summary()))

print('Saved:', eda_coef_out)
print('Saved:', eda_summary_out)
eda_coef_table

Saved: /Users/talia/Desktop/2026 S1/ECC3479/ecc3479-project/output/panel_eda_style_coefficients.csv
Saved: /Users/talia/Desktop/2026 S1/ECC3479/ecc3479-project/output/panel_eda_style_summary.txt


,coef,std_err,p_value,ci_low,ci_high
slope_2020,0.5850,0.0354,0.0000,0.5156,0.6545
slope_2021,0.6070,0.0354,0.0000,0.5377,0.6763
slope_2022,0.6292,0.0360,0.0000,0.5587,0.6997
slope_2023,0.7171,0.0376,0.0000,0.6433,0.7908
slope_2024,0.7720,0.0361,0.0000,0.7012,0.8429


## Economic Interpretation (EDA-Style Pooled Model)

1. **Strong positive correlation across all years.** The year-specific `ai_exposure` slopes are all positive and statistically significant in every year from 2020 to 2024, ranging from approximately 0.585 (2020) to 0.772 (2024). At the cross-sectional level, occupations with higher AI exposure are consistently associated with higher wages, and this differential appears to widen over the sample period.

2. **Descriptive, not causal.** Because the model contains no individual fixed effects, the estimated slopes absorb all time-invariant individual and occupational characteristics — including ability, educational sorting, and occupational prestige. The positive relationship between AI exposure and wages is therefore likely to reflect, at least in part, the fact that high-AI occupations have historically attracted higher-skilled and higher-paid workers. This model should be interpreted as a descriptive upper bound on the association, not as evidence of a causal wage effect of AI exposure.

3. **Role in the analysis.** This pooled model serves as a descriptive benchmark against the TWFE specification. Comparing the two isolates how much of the raw correlation survives once individual-level time-invariant heterogeneity is removed. The substantial attenuation from pooled to TWFE estimates suggests that occupational sorting is a meaningful confound in the raw relationship.

## Comparison with TWFE

Placing the pooled model alongside the TWFE results clarifies what each is measuring:

1. **The pooled model captures total association.** By including variation both across individuals and over time, it picks up long-run occupational wage differentials as well as any time-varying premium. This produces larger and more precisely estimated `ai_exposure` slopes, but these estimates conflate the causal effect of AI exposure with pre-existing differences in occupation quality, worker skill composition, and industry structure.

2. **The TWFE model isolates within-person variation.** Individual fixed effects remove all time-invariant personal characteristics, leaving only the question of whether the same person's wage changes when the return to their occupation's AI exposure changes over time. The resulting estimates are smaller and more conservative — by design, they cannot be driven by cross-sectional sorting.

3. **The gap between the two estimates reflects the role of selection.** The substantial difference between pooled slopes (~0.6–0.8) and TWFE interaction coefficients (significant only in 2021 at ~0.08) indicates that a large share of the raw correlation is attributable to compositional differences rather than AI-driven wage dynamics. High-AI-exposure occupations pay more partly because they attract higher-ability, better-educated workers — not solely because of AI.

4. **Recommended framing.** The pooled model is appropriate for the descriptive and EDA sections as evidence that "high-AI-exposure occupations command a wage premium." The TWFE is the primary identification result and should be the basis for any causal claim. Any discussion of the AI wage effect should be grounded in the TWFE estimates, with the pooled results cited only as descriptive context.